# Advanced AI - Assignment 7: Forest Variants

Work through the sections below in order. The implementations live in `advanced_ai/forest/isolation_forest.py`, `advanced_ai/forest/extra_trees.py`, and `advanced_ai/forest/quantile_forest.py`, and use NumPy only for the core algorithms.

## Core Equations And Rules

Use these definitions consistently in your implementation.

- Isolation tree split: pick feature $j$ uniformly at random, then pick threshold $t$ uniformly at random between the min and max of feature $j$ at the current node.
- Path length $h(x)$: number of edges from the root to the leaf containing $x$.
- Average path length normalizer over $\psi$ nodes:

$$
c(\psi) = 2H(\psi-1) - \frac{2(\psi-1)}{\psi}, \qquad H(i) \approx \ln(i) + 0.5772.
$$

- Anomaly score:

$$
s(x,\psi) = 2^{-\,\mathbb{E}[h(x)]/c(\psi)}.
$$

- Extra-Trees split rule: for each candidate feature, draw *one* threshold uniformly at random (no exhaustive search); keep the feature with the best $\Delta I$ among those random draws.
- QRF leaf weight for training example $i$:

$$
w_i(x) = \frac{1}{B}\sum_{b=1}^{B} \frac{\mathbf{1}\{x_i \in L_b(x)\}}{\bigl|L_b(x)\bigr|}.
$$

- Weighted empirical CDF and quantile prediction:

$$
\hat{F}(y\mid x) = \sum_{i=1}^{n} w_i(x)\,\mathbf{1}\{y_i \le y\}, \qquad \hat{Q}_\tau(x) = \inf\{y : \hat{F}(y\mid x) \ge \tau\}.
$$

## Step 1: Data files and preparation

Prepare the instructor-provided data bundles before running the notebook:

```bash
bash scripts/prepare_all_data.sh
bash scripts/check_prepare_all_data.sh
```

Expected prepared files:
- `datasets/isolation_toy.npz`
- `datasets/extra_trees_iris.npz`
- `datasets/quantile_regression.npz`

The scripts use the conda environment `ai_courses`.

In [ ]:
from advanced_ai.config import *
from advanced_ai.task_utils import require_file
from advanced_ai.data_utils import load_isolation_bundle, load_extra_trees_bundle, load_quantile_bundle
from advanced_ai.forest.isolation_forest import IsolationForest
from advanced_ai.forest.extra_trees import ExtraTreeClassifier, ExtraTreesClassifier
from advanced_ai.forest.quantile_forest import QuantileRegressionForest

print('Imports ready.')

## Roadmap

Work in this order:

1. Prepare and verify the datasets.
2. Complete the hand calculations for the path-length normalizer, anomaly score, and quantile prediction.
3. Implement the forest variants core in `advanced_ai/forest/isolation_forest.py`, `extra_trees.py`, and `quantile_forest.py`.
4. Run the self-checks.
5. Run the toy isolation, Extra-Trees, and quantile forest task runners.

Do not skip the self-checks before running the experiments.

## Step 2: Warm-up by hand

1. For $\psi=16$: $H(15) \approx 3.3182$. Compute $c(16)$ by hand.
2. Point $A$ has $\mathbb{E}[h(A)] = 2.0$ and point $B$ has $\mathbb{E}[h(B)] = 6.0$. Compute $s(A,16)$ and $s(B,16)$, and state which point is the anomaly.
3. A new point falls into leaves containing the pooled $y$-values (sorted) $\{12, 14, 15, 15, 16, 17, 17, 18, 19, 22\}$. Compute the median ($\tau=0.5$) prediction and the $80\%$ prediction interval ($\tau=0.1$, $\tau=0.9$).

## Step 3: Implement the Forest Variants core

Complete these TODOs in order. You may reuse your Lab 5 decision tree implementation and your Lab 6 random forest implementation; only the pieces below are new.

Implementation policy (strict):
- Implement the isolation tree, the random-threshold split rule, and the quantile aggregation with NumPy only.
- Do not use a library implementation (e.g. `IsolationForest`, `ExtraTreesClassifier`) for the main task.
- Plotting libraries are allowed for visual inspection.

### T1: Isolation tree growth

Open `advanced_ai/forest/isolation_forest.py` and implement `_build_node`.

- At each node, pick a feature $j$ uniformly at random, then a threshold $t$ uniformly at random between the min and max of feature $j$ at that node.
- Stop when a node has one point left, or the height limit $\approx \log_2 \psi$ is reached.
- If every feature is constant at the current node, stop and create a leaf.

### T2: Path length

Implement `path_length`.

Route $x$ from the root and count the number of edges to its leaf.

### T3: Anomaly score

Implement `_avg_path_length` (computes $c(\psi)$) and `anomaly_score`.

Average $h(x)$ over the $B$ isolation trees, then apply $s(x,\psi) = 2^{-\mathbb{E}[h(x)]/c(\psi)}$.

### T4: Extra-Trees split rule

Open `advanced_ai/forest/extra_trees.py` and implement `_random_threshold_split`.

- For each candidate feature, draw one threshold uniformly at random instead of searching all thresholds.
- Keep the feature whose random threshold gives the largest $\Delta I$.
- Reuse your Lab 5 impurity and Lab 6 bagging loop; only the threshold selection changes.

### T5: Leaf distribution collection

Open `advanced_ai/forest/quantile_forest.py` and implement `_collect_leaf_values`.

For a regression forest trained as in Lab 6, store the training $y$-values that reach each leaf, for every tree.

### T6: Weighted quantile prediction

Implement `predict_quantile`.

- Compute $w_i(x)$ from the leaves $x$ falls into across all $B$ trees.
- Build $\hat{F}(y\mid x)$ and return $\hat{Q}_\tau(x) = \inf\{y : \hat{F}(y\mid x) \ge \tau\}$.

## Step 4: Run the self-checks

After implementing the core classes, run:

```bash
python -m advanced_ai.self_checks
```

Continue only after every check passes.

## Step 5: Toy Isolation Forest

Run:

```bash
python toy_isolation.py
```

Deliverables:
- average path length and anomaly score for an injected outlier and for a normal point;
- saved anomaly score plot (score versus normalized path length, as in the lecture);
- the threshold used to flag anomalies and how many points were flagged.

## Step 6: Extra-Trees vs Random Forest on Iris

Run:

```bash
python extra_trees_iris.py
```

Deliverables:
- training time, training accuracy, and test accuracy for each $B \in \{1, 5, 10, 20, 50, 100\}$ and each method;
- which method trains faster and which has lower variance across repeated runs.

## Step 7: Quantile Regression Forest

Run:

```bash
python quantile_forest_demo.py
```

Deliverables:
- median prediction and an $80\%$ prediction interval ($\tau=0.1,0.9$) for three test points;
- saved plot of the predicted median curve with the prediction interval band;
- the fraction of held-out test points that actually fall inside their predicted $80\%$ interval.

## Step 8: Optional extension

Complete at most one optional extension after finishing the required work.

- Vary the subsample size $\psi \in \{64, 128, 256, 512\}$ and report how the anomaly scores change.
- Try a $50\%$ and a $95\%$ prediction interval instead of $80\%$ and compare interval width.
- Combine Extra-Trees splitting with the quantile-forest leaf aggregation and report whether the prediction intervals change.

## Submission checklist

Submit:
- completed `forest_variants.ipynb`;
- completed `advanced_ai/forest/isolation_forest.py`, `extra_trees.py`, and `quantile_forest.py`;
- generated files from `results/`;
- one PDF report with hand calculations, plots, tables, and conclusions;
- reproducibility notes including random seed and environment.